In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import numpy as np
import pandas as pd
import os
from transformers import AutoModel, AutoImageProcessor
from tqdm import tqdm
from PIL import Image
from dotenv import load_dotenv

from transformers.utils import logging
logging.disable_progress_bar()

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

In [3]:
model_id = "openai/clip-vit-large-patch14"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

print(f"using {device} with {dtype}")

using cuda with torch.bfloat16


In [4]:
processor = AutoImageProcessor.from_pretrained(model_id, token=HF_TOKEN)
model = AutoModel.from_pretrained(
    model_id,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    output_hidden_states=True,
    token=HF_TOKEN
).to(device).eval()
print("model loaded")

`torch_dtype` is deprecated! Use `dtype` instead!
CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model loaded


In [5]:
DEBIAS_LAYER = 15 # Intervention layer
IMAGES_DIR = "data/images"
METADATA_PATH = "data/metadata.csv"
ACTIVATIONS_DIR = "data/activations"

os.makedirs(ACTIVATIONS_DIR, exist_ok=True)

df = pd.read_csv(METADATA_PATH)

In [6]:
def extract_base_activations(df_subset, layer_idx):
    activations = []
    labels = []
    captured_activation = None
    
    def hook_fn(module, input, output):
        nonlocal captured_activation
        is_tuple = isinstance(output, tuple)
        hidden_states = output[0] if is_tuple else output
        # Extract [CLS] token at index 0
        captured_activation = hidden_states[:, 0, :].detach().cpu()

    hook_handle = model.vision_model.encoder.layers[layer_idx].register_forward_hook(hook_fn)
    
    try:
        with torch.no_grad():
            for _, row in tqdm(df_subset.iterrows(), total=len(df_subset)):
                img_path = os.path.join(IMAGES_DIR, row['filename'])
                img = Image.open(img_path).convert("RGB")
                inputs = processor(images=img, return_tensors="pt").to(device, dtype=dtype)
                
                model.get_image_features(**inputs)
                activations.append(captured_activation.float().numpy().squeeze())
                labels.append(row['bald'])
    finally:
        hook_handle.remove()
        
    return np.array(activations), np.array(labels)

In [7]:
# We only need baseline activations for the training set at DEBIAS_LAYER to build CAVs
train_df = df[df['split'] == 'train']

print(f"Extracting baseline activations for {len(train_df)} train images at layer {DEBIAS_LAYER}...")
acts_train, labels_train = extract_base_activations(train_df, DEBIAS_LAYER)

# Save to CSV for the next step
base_csv_path = os.path.join(ACTIVATIONS_DIR, f"base_acts_train_layer_{DEBIAS_LAYER}.csv")
acts_df = pd.DataFrame(acts_train)
acts_df['label'] = labels_train
acts_df.to_csv(base_csv_path, index=False)
print(f"Saved {acts_train.shape} activations to {base_csv_path}")

Extracting baseline activations for 2000 train images at layer 15...


100%|██████████| 2000/2000 [06:36<00:00,  5.04it/s]


Saved (2000, 1024) activations to data/activations\base_acts_train_layer_15.csv
